# Concatenate DP-SCREAM Output Variable

This notebook extracts a specified variable from multiple DP-SCREAM history files and concatenates them into a single output file.

**Variable:** `LW_flux_up_at_model_top`  
**Source:** `scream_cpu_dpxx_RCE_dx1km` simulation run output (AVERAGE, 5-min interval)  
**Period:** 2000-01-01-00000 to 2000-01-25-83100


In [1]:
import os
import glob
import xarray as xr
import numpy as np
import ctypes, ctypes.util

# Suppress benign HDF5 "file not found" diagnostics printed to stderr
_hdf5_lib = ctypes.util.find_library("hdf5")
if _hdf5_lib:
    _ = ctypes.CDLL(_hdf5_lib).H5Eset_auto2(0, None, None)

import warnings
warnings.filterwarnings("ignore")


In [ ]:

def extract_timestamp(filepath):
    """Return the YYYY-MM-DD-sssss timestamp string from a filename."""
    basename = os.path.basename(filepath)
    # Remove prefix and suffix to isolate timestamp
    ts = basename[len(file_prefix):-len(file_suffix)]
    return ts


In [10]:
# --- CONFIGURATION ---
icase      = "scream_cpu_dpxx_RCE_dx1km"
run_dir    = f"/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/{icase}/run"
out_dir    = f"/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/{icase}/processed"

varname    = "LW_flux_up_at_model_top"

# File naming parameters
file_prefix  = f"{icase}.hist.AVERAGE.nmins_x5."
file_suffix  = ".nc"

# Date-range timestamps (inclusive) matching filename format YYYY-MM-DD
ts_start = "2000-01-01"
ts_end   = "2000-01-10"


In [11]:

styear = int(ts_start[:4])
stmonth = int(ts_start[5:7])
stday = int(ts_start[8:10])

edyear = int(ts_end[:4])
edmonth = int(ts_end[5:7])
edday = int(ts_end[8:10])

print(f"Variable  : {varname}")
print(f"Run dir   : {run_dir}")
print(f"Output dir: {out_dir}")
print(f"Period    : {ts_start}  to  {ts_end}")


Variable  : LW_flux_up_at_model_top
Run dir   : /pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run
Output dir: /pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/processed
Period    : 2000-01-01  to  2000-01-10


In [12]:
# --- BUILD FILE LIST ---
# Collect all matching AVERAGE history files and filter to the requested date range.
# Filenames sort lexicographically (zero-padded YYYY-MM-DD-sssss) so sorted() is sufficient.

all_files = sorted(glob.glob(os.path.join(run_dir, file_prefix + "*.nc")))


some data are saved with the file name date stamp of the previous and next days
So better to include the day before stday (if file exist) and the day after the edday

In [15]:
#create day arrays with numpy datetime64
start_date = np.datetime64(f"{styear:04d}-{stmonth:02d}-{stday:02d}")
end_date = np.datetime64(f"{edyear:04d}-{edmonth:02d}-{edday:02d}")

date_range = np.arange(start_date, end_date + np.timedelta64(1, 'D'), dtype='datetime64[D]')

ndays = len(date_range)

In [17]:
iday = 0
day_str = str(date_range[iday])  # Format: 'YYYY-MM-DD'
print(f"Processing day {iday+1}/{ndays}: {day_str}")

# Filter files for this day
day_files = sorted(glob.glob(os.path.join(run_dir, file_prefix + day_str + "-*.nc")))
#day_files = [f for f in selected_files if extract_timestamp(f).startswith(day_str)]
day_files

Processing day 1/10: 2000-01-01


['/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-00000.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-03600.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-07200.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-10800.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-14400.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-18000.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVE

In [18]:
previous_day = date_range[iday] - np.timedelta64(1, 'D')
previous_day_str = str(previous_day)
previous_day_files = sorted(glob.glob(os.path.join(run_dir, file_prefix + previous_day_str + "-*.nc")))
previous_day_file = previous_day_files[-1] if previous_day_files else None
previous_day_file

In [21]:
if(previous_day_file):
    print(f"  Checking any outputs in the previous day's last file: {previous_day_file}")
else:
    print(f"  No previous day file found, skipping check.")

  No previous day file found, skipping check.


In [22]:
next_day = date_range[iday] + np.timedelta64(1, 'D')
next_day_str = str(next_day)
next_day_files = sorted(glob.glob(os.path.join(run_dir, file_prefix + next_day_str + "-*.nc")))
next_day_file = next_day_files[0] if next_day_files else None
next_day_file

'/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-00300.nc'

In [ ]:
if(next_day_file):
    print(f"  Checking any outputs in the next day's first file: {next_day_file}")
    # Open and concatenate files for this day
    ds_next = xr.open_dataset(next_day_file)
    time_nextday = ds_next['time'].values
    print(f"    Next day file time range: {time_nextday[0]} to {time_nextday[-1]}")
else:
    print(f"  No next day file found, skipping check.")

  Checking any outputs in the next day's first file: /pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-00300.nc


In [ ]:
time_nextday = ds_next['time'].values
print(f"    Next day file time range: {time_nextday[0]} to {time_nextday[-1]}")


    Next day file time range: 2000-01-02 00:10:00 to 2000-01-02 01:05:00


In [ ]:

if not day_files:
    print(f"  No files found for {day_str}, skipping.")

print(f"  Found {len(day_files)} files for {day_str}, opening...")

# Open and concatenate files for this day
ds_day = xr.open_mfdataset(day_files, combine='by_coords', parallel=True)[[varname]]


In [16]:
date_range
all_files

array(['2000-01-01', '2000-01-02', '2000-01-03', '2000-01-04',
       '2000-01-05', '2000-01-06', '2000-01-07', '2000-01-08',
       '2000-01-09', '2000-01-10'], dtype='datetime64[D]')

['/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-00000.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-03600.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-07200.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-10800.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-14400.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-18000.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVE

In [6]:

# Keep only files whose timestamp falls in [ts_start, ts_end]
selected_files = [
    f for f in all_files
    if ts_start <= extract_timestamp(f) <= ts_end
]

print(f"Total AVERAGE files found : {len(all_files)}")
print(f"Files in selected range   : {len(selected_files)}")
if selected_files:
    print(f"First file : {os.path.basename(selected_files[0])}")
    print(f"Last file  : {os.path.basename(selected_files[-1])}")


Total AVERAGE files found : 625
Files in selected range   : 241
First file : scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-01-00000.nc
Last file  : scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-10-83100.nc


In [7]:
# --- EXTRACT AND CONCATENATE ---
# Open all selected files with open_mfdataset, keeping only the target variable
# (plus time coordinate). This avoids loading unnecessary data into memory.

if not selected_files:
    raise RuntimeError("No files found in the specified date range. Check run_dir and date range.")


In [18]:
iday = 1
day_str = str(date_range[iday])  # Format: 'YYYY-MM-DD'
print(f"Processing day {iday+1}/{ndays}: {day_str}")

# Filter files for this day
day_files = [f for f in selected_files if extract_timestamp(f).startswith(day_str)]

if not day_files:
    print(f"  No files found for {day_str}, skipping.")

print(f"  Found {len(day_files)} files for {day_str}, opening...")

# Open and concatenate files for this day
ds_day = xr.open_mfdataset(day_files, combine='by_coords', parallel=True)[[varname]]


Processing day 2/11: 2000-01-02
  Found 24 files for 2000-01-02, opening...


In [23]:
day_files

['/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-00300.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-03900.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-07500.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-11100.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-14700.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVERAGE.nmins_x5.2000-01-02-18300.nc',
 '/pscratch/sd/k/ksa/simulation/DP-SCREAM/cases/scream_cpu_dpxx_RCE_dx1km/run/scream_cpu_dpxx_RCE_dx1km.hist.AVE

In [19]:
day_str

'2000-01-02'

In [22]:
itime = ds_day['time']
print(f"  Time coordinate values: {itime.values}")

  Time coordinate values: [cftime.DatetimeNoLeap(2000, 1, 2, 0, 10, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 15, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 20, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 25, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 30, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 35, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 40, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 45, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 50, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 0, 55, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 1, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 1, 5, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 1, 10, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 2, 1, 15, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2

In [20]:
ds_day_only = ds_day.sel(time=ds_day.time.dt.strftime("%Y-%m-%d") == day_str)

In [21]:
ds_day_only

<xarray.Dataset> Size: 183MB
Dimensions:                  (time: 286, ncol: 160000)
Coordinates:
  * time                     (time) object 2kB 2000-01-02 00:10:00 ... 2000-0...
Dimensions without coordinates: ncol
Data variables:
    LW_flux_up_at_model_top  (time, ncol) float32 183MB dask.array<chunksize=(12, 160000), meta=np.ndarray>
Attributes: (12/22)
    case_t0:                      2000-01-01-00000
    run_t0:                       2000-01-02-00000
    averaging_type:               AVERAGE
    averaging_frequency_units:    nmins
    averaging_frequency:          5
    file_max_storage_type:        num_snapshots
    ...                           ...
    contact:                      e3sm-data-support@llnl.gov
    institution_id:               E3SM-Project
    realm:                        atmos
    history:                      created on Mon Mar 16 20:19:42 2026
    Conventions:                  CF-1.8
    product:                      model-output

In [17]:
ds_day.close()
del ds_day

In [16]:
ds_day_only.close()
del ds_day_only

In [12]:
itime = ds_day['time']
print(f"  Time coordinate values: {itime.values}")

  Time coordinate values: [cftime.DatetimeNoLeap(2000, 1, 1, 0, 5, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 10, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 15, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 20, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 25, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 30, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 35, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 40, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 45, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 50, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 0, 55, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 1, 0, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 1, 5, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(2000, 1, 1, 1, 10, 0, 0, has_year_zero=True)
 cftime.DatetimeNoLeap(20

In [10]:

# Save concatenated dataset for this day
out_path = os.path.join(out_dir, f"{icase}.{varname}.hist.AVERAGE.{day_str}.nc")
ds_day.to_netcdf(out_path)
print(f"  Saved concatenated data to {out_path}")

NameError: name 'ds_day' is not defined

In [ ]:
for iday in range(ndays):
    day_str = str(date_range[iday])  # Format: 'YYYY-MM-DD'
    print(f"Processing day {iday+1}/{ndays}: {day_str}")
    
    # Filter files for this day
    day_files = [f for f in selected_files if extract_timestamp(f).startswith(day_str)]
    
    if not day_files:
        print(f"  No files found for {day_str}, skipping.")
        continue
    
    print(f"  Found {len(day_files)} files for {day_str}, opening...")
    
    # Open and concatenate files for this day
    ds_day = xr.open_mfdataset(day_files, combine='by_coords', parallel=True)[[varname]]
    
    # Save concatenated dataset for this day
    out_path = os.path.join(out_dir, f"{icase}.{varname}.hist.AVERAGE.{day_str}.nc")
    ds_day.to_netcdf(out_path)
    print(f"  Saved concatenated data to {out_path}")

In [ ]:

print(f"Opening {len(selected_files)} files with xarray.open_mfdataset ...")

ds_concat = xr.open_mfdataset(
    selected_files,
    combine="by_coords",
    data_vars="minimal",    # only load variables that differ across files
    coords="minimal",       # only load coordinates that differ across files
    compat="override",      # skip expensive coordinate agreement checks
    parallel=False,         # set True if Dask is available
)[varname].to_dataset()

print("Concatenation complete.")
print(ds_concat)


In [ ]:
# --- SAVE OUTPUT ---
os.makedirs(out_dir, exist_ok=True)

out_filename = f"{icase}.{varname}.{ts_start}_to_{ts_end}.nc"
out_path = os.path.join(out_dir, out_filename)

print(f"Writing output to:\n  {out_path}")

ds_concat.to_netcdf(out_path)

print("Done.")


In [ ]:
# --- VERIFY OUTPUT ---
ds_check = xr.open_dataset(out_path)
print("Output file contents:")
print(ds_check)
print(f"\n'{varname}' shape : {ds_check[varname].shape}")
print(f"time range        : {ds_check['time'].values[0]}  to  {ds_check['time'].values[-1]}")
ds_check.close()
